# Manual Image Generator

Enter a prompt yourself and generate images with SDXL — no prompt-generator API involved.

Fill in the config cell below, then run all cells in order.

In [ ]:
# Cell 1: Setup & Configuration (Manual)
import os
from dotenv import load_dotenv
from google.colab import drive
import warnings

warnings.filterwarnings('ignore', message='Flax classes are deprecated')
warnings.filterwarnings('ignore', category=FutureWarning)

# Mount Google Drive
mount_path = '/content/drive'
if not os.path.exists(mount_path):
    print("Mounting Google Drive...")
    drive.mount(mount_path)
else:
    print("Drive already mounted.")

# Load HuggingFace token
env_path = '/content/drive/MyDrive/AI/hf_token.env'
load_dotenv(env_path)
huggingface_token = os.getenv('HUGGINGFACE_TOKEN')

#@markdown ### Model
model_id = "SG161222/RealVisXL_V5.0" #@param {type:"string"}
download_model = False #@param {type:"boolean"}

#@markdown ### Prompt (edit these directly — no API calls)
prompt = "a photo of a mountain landscape at sunset, dramatic lighting" #@param {type:"string"}
prompt_2 = "" #@param {type:"string"}
negative_prompt = "text, writing, bad teeth, deformed face, child, childish, young, deformed, extra fingers" #@param {type:"string"}

#@markdown ### Generation Settings
num_images = 4 #@param {type:"integer"}
scheduler_name = "DPM++ 2M" #@param ["Model Default", "DPM++ 2M", "DPM++ 2M Karras", "DPM++ SDE", "DPM++ SDE Karras", "Euler", "Euler a", "Heun", "KDPM2", "KDPM2 a", "LMS", "DDIM", "PNDM", "UniPC"]
guidance_scale = 7.5 #@param {type:"slider", min:1, max:20, step:0.5}
num_steps = 35 #@param {type:"slider", min:10, max:80, step:1}
width = 1024 #@param [1024, 1152, 896, 1216, 832, 1344, 768]
height = 1024 #@param [1024, 896, 1152, 832, 1216, 768, 1344]
seed = -1 #@param {type:"integer"}

#@markdown ### LoRA (optional)
lora_enabled = False #@param {type:"boolean"}
lora_path = "Loras/stoo_tee/output/stoo_tee.safetensors" #@param {type:"string"}
lora_strength = 0.8 #@param {type:"slider", min:0.1, max:1.5, step:0.05}
lora_trigger_word = "stoo_tee" #@param {type:"string"}
lora_prepend_trigger = True #@param {type:"boolean"}

if lora_enabled and lora_prepend_trigger and lora_trigger_word and lora_trigger_word not in prompt:
    prompt = f"{lora_trigger_word}, {prompt}"
    if prompt_2:
        prompt_2 = f"{lora_trigger_word}, {prompt_2}"

lora_full_path = f"/content/drive/MyDrive/{lora_path}" if lora_enabled else None

from datetime import datetime
base_path = "/content/drive/MyDrive/AI/"
model_path = base_path + "models/" + model_id
save_directory = f"{base_path}images/{datetime.now().strftime('%Y%m%d%H%M%S')}/"
os.makedirs(save_directory, exist_ok=True)

print(f"\n=== Configuration ===")
print(f"Model: {model_id}")
print(f"Prompt: {prompt}")
if prompt_2:
    print(f"Prompt 2: {prompt_2}")
print(f"Negative: {negative_prompt}")
print(f"Images: {num_images} | Scheduler: {scheduler_name} | Steps: {num_steps} | Guidance: {guidance_scale}")
print(f"Resolution: {width}x{height}")
if lora_enabled:
    print(f"LoRA: {lora_path} @ {lora_strength}")
print(f"Save directory: {save_directory}")

In [ ]:
# Cell 2: Install & Import Dependencies
!pip install -q diffusers transformers accelerate safetensors compel
!pip install -q -U "peft>=0.15.0" "torchao>=0.16.0"

import torch
import random
import json
import gc
from PIL import Image
from huggingface_hub import snapshot_download
from diffusers import (
    StableDiffusionXLPipeline,
    AutoencoderKL,
    DPMSolverMultistepScheduler,
    DPMSolverSinglestepScheduler,
    EulerDiscreteScheduler,
    EulerAncestralDiscreteScheduler,
    KDPM2DiscreteScheduler,
    KDPM2AncestralDiscreteScheduler,
    HeunDiscreteScheduler,
    LMSDiscreteScheduler,
    DDIMScheduler,
    PNDMScheduler,
    UniPCMultistepScheduler
)
from compel import CompelForSDXL

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.mem_get_info()[0] / 1024**3:.2f}GB free")

In [ ]:
# Cell 3: Load Model & Pipeline
model_exists = os.path.exists(model_path) and os.path.isdir(model_path) and len(os.listdir(model_path)) > 0

if download_model or not model_exists:
    print(f"Downloading model {model_id}...")
    snapshot_download(
        repo_id=model_id,
        local_dir=model_path,
        token=huggingface_token if huggingface_token else None,
        ignore_patterns=["*.safetensors.lock"]
    )
    print(f"✓ Model downloaded")
else:
    print(f"✓ Model exists at {model_path}")

# Load pipeline
print(f"\nLoading pipeline...")
load_kwargs = {
    "torch_dtype": torch.float16,
    "use_safetensors": True,
}

pipe = None
for attempt in ["fp16", "no_variant"]:
    try:
        if attempt == "fp16":
            load_kwargs["variant"] = "fp16"
        else:
            load_kwargs.pop("variant", None)

        try:
            pipe = StableDiffusionXLPipeline.from_pretrained(model_path, **load_kwargs)
            print(f"✓ Loaded from local path")
        except (OSError, FileNotFoundError):
            load_kwargs["token"] = huggingface_token
            pipe = StableDiffusionXLPipeline.from_pretrained(model_id, **load_kwargs)
            print(f"✓ Loaded from HuggingFace")
        break
    except ValueError as e:
        if "variant" in str(e) and attempt == "fp16":
            continue
        raise

# Load enhanced VAE
print("Loading enhanced VAE...")
try:
    vae = AutoencoderKL.from_pretrained(
        "madebyollin/sdxl-vae-fp16-fix",
        torch_dtype=torch.float16
    )
    pipe.vae = vae
    print("✓ Enhanced VAE applied")
except Exception as e:
    print(f"⚠ Could not load enhanced VAE: {e}")

# Move to GPU and optimize (each of these is a memory optimization only —
# none should be able to abort the run if a diffusers version lacks it)
pipe = pipe.to("cuda")
for opt_name in ("enable_attention_slicing", "enable_vae_slicing", "enable_vae_tiling"):
    try:
        getattr(pipe, opt_name)()
        print(f"✓ {opt_name} enabled")
    except AttributeError:
        print(f"⚠ {opt_name} not available in this diffusers version, skipping")

try:
    pipe.enable_xformers_memory_efficient_attention()
    print("✓ xformers enabled")
except:
    print("⚠ xformers not available")

# Load LoRA if enabled (kept unfused — strength applied via cross_attention_kwargs at inference)
if lora_enabled and lora_full_path:
    if os.path.exists(lora_full_path):
        print(f"Loading LoRA from {lora_full_path}...")
        pipe.load_lora_weights(lora_full_path)
        print(f"✓ LoRA loaded @ strength {lora_strength}")
        print(f"  Trigger word: '{lora_trigger_word}' — already prepended to prompt above if enabled")
    else:
        print(f"✗ LoRA file not found: {lora_full_path}")
        lora_enabled = False

# Initialize Compel for prompt weighting (single-encoder mode only)
compel = CompelForSDXL(pipe)
print("✓ Compel initialized")

print(f"\n✓ Pipeline ready! GPU Memory: {torch.cuda.mem_get_info()[0] / 1024**3:.2f}GB free")

In [ ]:
# Cell 4: Initialize Scheduler
def initialize_samplers(base_config):
    """Create scheduler instances from model's config."""
    clean_config = {k: v for k, v in base_config.items() if k not in ['beta_schedule']}

    return {
        "Model Default": lambda: pipe.scheduler.__class__.from_config(base_config),
        "DPM++ 2M": lambda: DPMSolverMultistepScheduler.from_config(base_config),
        "DPM++ 2M Karras": lambda: DPMSolverMultistepScheduler.from_config(clean_config, use_karras_sigmas=True),
        "DPM++ SDE": lambda: DPMSolverSinglestepScheduler.from_config(base_config),
        "DPM++ SDE Karras": lambda: DPMSolverSinglestepScheduler.from_config(clean_config, use_karras_sigmas=True),
        "Euler": lambda: EulerDiscreteScheduler.from_config(base_config),
        "Euler a": lambda: EulerAncestralDiscreteScheduler.from_config(base_config),
        "Heun": lambda: HeunDiscreteScheduler.from_config(base_config),
        "KDPM2": lambda: KDPM2DiscreteScheduler.from_config(base_config),
        "KDPM2 a": lambda: KDPM2AncestralDiscreteScheduler.from_config(base_config),
        "LMS": lambda: LMSDiscreteScheduler.from_config(base_config),
        "DDIM": lambda: DDIMScheduler.from_config(base_config),
        "PNDM": lambda: PNDMScheduler.from_config(base_config),
        "UniPC": lambda: UniPCMultistepScheduler.from_config(base_config),
    }

SAMPLERS = initialize_samplers(pipe.scheduler.config)

if scheduler_name not in SAMPLERS:
    print(f"\u26a0 Unknown scheduler '{scheduler_name}', defaulting to Euler")
    scheduler_name = "Euler"

pipe.scheduler = SAMPLERS[scheduler_name]()
print(f"\u2713 Scheduler set to {scheduler_name}")

In [ ]:
# Cell 5: Generation Loop
metadata_list = []
metadata_path = os.path.join(save_directory, "metadata.json")

print(f"Generating {num_images} image(s) with prompt:\n  {prompt}\n")

for i in range(num_images):
    try:
        image_seed = seed if seed >= 0 else random.randint(0, 2**32 - 1)
        # Vary the seed per image when a fixed seed was given, so images differ
        if seed >= 0:
            image_seed = seed + i
        generator = torch.Generator(device="cuda").manual_seed(image_seed)

        lora_kwargs = {}
        if lora_enabled and lora_full_path:
            lora_kwargs["cross_attention_kwargs"] = {"scale": lora_strength}

        if prompt_2:
            # Dual-encoder mode: pass separate prompts directly
            result = pipe(
                prompt=prompt,
                prompt_2=prompt_2,
                negative_prompt=negative_prompt,
                width=width,
                height=height,
                guidance_scale=guidance_scale,
                num_inference_steps=num_steps,
                generator=generator,
                **lora_kwargs,
            ).images[0]
        else:
            # Single prompt mode: use Compel for long-prompt weighting
            conditioning = compel(prompt, negative_prompt=negative_prompt)
            result = pipe(
                prompt_embeds=conditioning.embeds,
                pooled_prompt_embeds=conditioning.pooled_embeds,
                negative_prompt_embeds=conditioning.negative_embeds,
                negative_pooled_prompt_embeds=conditioning.negative_pooled_embeds,
                width=width,
                height=height,
                guidance_scale=guidance_scale,
                num_inference_steps=num_steps,
                generator=generator,
                **lora_kwargs,
            ).images[0]

        filename = f"{datetime.now().strftime('%Y%m%d%H%M%S')}_{str(i).zfill(4)}_{scheduler_name.replace(' ', '_').replace('+', 'p')}.png"
        result.save(os.path.join(save_directory, filename))

        metadata = {
            "filename": filename,
            "model": model_id,
            "scheduler": scheduler_name,
            "prompt": prompt,
            "prompt_2": prompt_2,
            "negative_prompt": negative_prompt,
            "seed": image_seed,
            "width": width,
            "height": height,
            "guidance_scale": guidance_scale,
            "num_steps": num_steps,
            "lora_enabled": lora_enabled,
            "lora_path": lora_path if lora_enabled else "",
            "lora_strength": lora_strength if lora_enabled else 0,
        }
        metadata_list.append(metadata)

        with open(metadata_path, 'w') as f:
            json.dump(metadata_list, f, indent=2)

        del result
        torch.cuda.empty_cache()

        print(f"[{i+1}/{num_images}] seed={image_seed} saved as {filename}")

    except Exception as e:
        print(f"Error generating image {i+1}: {e}")
        continue

print(f"\n✓ Generation complete!")
print(f"✓ {len(metadata_list)} images saved to: {save_directory}")

In [ ]:
# Cell 6: Display Results
import matplotlib.pyplot as plt

num_columns = 3
max_images = 12

if metadata_list:
    display_meta = metadata_list[:max_images]
    num_rows = (len(display_meta) + num_columns - 1) // num_columns

    plt.figure(figsize=(5 * num_columns, 5 * num_rows))

    for idx, meta in enumerate(display_meta):
        img = Image.open(os.path.join(save_directory, meta['filename']))
        plt.subplot(num_rows, num_columns, idx + 1)
        plt.imshow(img)
        plt.axis('off')
        plt.title(f"seed={meta['seed']}\n{meta['width']}x{meta['height']}", fontsize=8)

    plt.tight_layout()
    plt.show()
    print(f"Displayed {len(display_meta)} of {len(metadata_list)} images")
else:
    print("No images to display")

In [ ]:
# Cell 7: Cleanup
del pipe, compel
torch.cuda.empty_cache()
gc.collect()

print("\u2713 Cleanup complete")
if torch.cuda.is_available():
    print(f"GPU Memory: {torch.cuda.mem_get_info()[0] / 1024**3:.2f}GB free")